# 04 — Anomaly Detection

Detect abnormal price spikes across Algerian food markets using:
- **Isolation Forest**
- **One-Class SVM**
- **Ensemble composite score**

Outputs: alert table, score visualisation, severity breakdown.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from src.data_ingestion.wfp_connector import WFPConnector
from src.preprocessing.data_cleaner import DataCleaner
from src.preprocessing.feature_engineering import FeatureEngineer
from src.models.anomaly_detector import AnomalyDetector
from src.utils.helpers import load_config

config = load_config('../config/config.yaml')

# Load and prepare data
raw = WFPConnector(config=config).fetch_price_data()
clean = DataCleaner(config=config).clean(raw)
features = FeatureEngineer(config=config).build_features(clean)
print(f'Features shape: {features.shape}')

In [ ]:
# ── Fit & Predict ─────────────────────────────────────────────
detector = AnomalyDetector(config=config)
detector.fit(features)
scored = detector.predict(features)

print(f"Total anomalies: {scored['is_anomaly'].sum():,}")
print(f"Anomaly rate: {scored['is_anomaly'].mean():.2%}")
scored[scored['is_anomaly']].head(10)

In [ ]:
# ── Score Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(scored['anomaly_score'], bins=50, color='#1565c0', edgecolor='white', alpha=0.85)
axes[0].axvline(detector.alert_threshold, color='red', linestyle='--', label=f'Seuil ({detector.alert_threshold})')
axes[0].set_title('Distribution des Scores d\'Anomalie')
axes[0].set_xlabel('Score')
axes[0].legend()

anomaly_by_product = scored.groupby('product')['is_anomaly'].sum().sort_values(ascending=False)
anomaly_by_product.plot(kind='barh', ax=axes[1], color='#d32f2f', alpha=0.8)
axes[1].set_title('Anomalies par Produit')
axes[1].set_xlabel('Nombre d\'Anomalies')

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise Anomalies on Price Series ──────────────────────
product = 'Tomates'
region  = 'Alger'

subset = scored[(scored['product'] == product) & (scored['region'] == region)].sort_values('date')
anomalies = subset[subset['is_anomaly']]

fig = go.Figure()
fig.add_trace(go.Scatter(x=subset['date'], y=subset['price'], name='Prix', line=dict(color='#006233')))
fig.add_trace(go.Scatter(
    x=anomalies['date'], y=anomalies['price'],
    mode='markers', name='Anomalie',
    marker=dict(color='red', size=10, symbol='x')
))
fig.update_layout(
    title=f'Anomalies de Prix — {product} / {region}',
    xaxis_title='Date', yaxis_title='Prix (DZD)',
    template='plotly_white', height=380
)
fig.show()

In [ ]:
# ── Generate & Display Alerts ────────────────────────────────
alerts = detector.generate_alerts(scored)
print(f'Generated {len(alerts)} alerts')

alerts_df = detector.alerts_to_dataframe()

# Severity breakdown
sev_counts = alerts_df['severity'].value_counts()
colors = {'CRITICAL': '#d32f2f', 'HIGH': '#f57c00', 'MEDIUM': '#fbc02d', 'LOW': '#388e3c'}
fig, ax = plt.subplots(figsize=(7, 4))
sev_counts.plot(kind='bar', ax=ax, color=[colors.get(s, 'grey') for s in sev_counts.index], rot=0)
ax.set_title('Alertes par Sévérité')
ax.set_ylabel('Nombre')
plt.tight_layout()
plt.show()

alerts_df.sort_values('anomaly_score', ascending=False).head(10)

In [ ]:
# ── Save model ────────────────────────────────────────────────
model_path = detector.save('anomaly_v1')
print(f'Model saved: {model_path}')